# TensorFlow Tutorial: From Beginner to Deep Learning

**Goal:** Learn TensorFlow and Keras step by step, from tensor operations and automatic differentiation to training, evaluating, saving, and deploying the basics of neural networks.

## Tutorial roadmap

1. Introduction and setup
2. TensorFlow tensors and operations
3. Automatic differentiation with `tf.GradientTape`
4. Build a neural network with Keras
5. Train a binary classifier on synthetic data
6. Evaluate and make predictions
7. Save and reload a model
8. GPU detection and device placement
9. Mini-project: handwritten digit classification with MNIST
10. Practice exercises and interview questions

**Suggested workflow:** Run the notebook cells from top to bottom. Training times and exact metrics can vary by TensorFlow version and hardware.

## 1. Introduction to TensorFlow

TensorFlow is an open-source machine learning framework. **Keras** is its high-level API for defining and training neural networks.

Common uses include image classification, text processing, forecasting, recommendation systems, and deep learning applications.

### Install the libraries

Run this cell if TensorFlow is not installed. You may need to restart the notebook kernel after installation.

In [ ]:
%pip install tensorflow matplotlib scikit-learn


### Import TensorFlow and check the environment

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))


## 2. Understanding tensors

A tensor is a data structure that can represent a scalar, vector, matrix, or higher-dimensional data. TensorFlow tensors are the basic objects used for numerical computation.

| Type | Example | Rank |
|---|---|---:|
| Scalar | `5` | 0 |
| Vector | `[1, 2, 3]` | 1 |
| Matrix | `[[1, 2], [3, 4]]` | 2 |
| Image batch | Multiple images | Usually 4 |

### Create tensors and inspect their properties

In [ ]:
scalar = tf.constant(5)
vector = tf.constant([1, 2, 3])
matrix = tf.constant([[1, 2], [3, 4]])
random_tensor = tf.random.uniform((3, 4))

print("Scalar:", scalar)
print("Vector:", vector)
print("Matrix:\n", matrix)
print("Random tensor:\n", random_tensor)
print("Matrix shape:", matrix.shape)
print("Matrix rank:", tf.rank(matrix).numpy())
print("Matrix dtype:", matrix.dtype)


### Tensor operations

In [ ]:
a = tf.constant([1.0, 2.0, 3.0])
b = tf.constant([4.0, 5.0, 6.0])

print("Addition:", a + b)
print("Subtraction:", a - b)
print("Elementwise multiplication:", a * b)
print("Division:", a / b)

x = tf.constant([[1.0, 2.0], [3.0, 4.0]])
y = tf.constant([[5.0, 6.0], [7.0, 8.0]])
print("Matrix multiplication:\n", tf.matmul(x, y))


### Reshape a tensor

In [ ]:
x = tf.range(12)
reshaped = tf.reshape(x, (3, 4))

print("Original:", x.numpy())
print("Reshaped:\n", reshaped.numpy())
print("Shape:", reshaped.shape)


**Key concept:** The shapes of tensors must be compatible with the operations you perform. TensorFlow eager execution usually evaluates operations immediately, and `.numpy()` converts a tensor to a NumPy value when running eagerly.

## 3. Automatic differentiation with `tf.GradientTape`

During training, a model needs gradients to understand how its parameters should change to reduce prediction error. TensorFlow's `tf.GradientTape` records operations and computes derivatives.

For

\[
y=x^2+3x+2,\qquad \frac{dy}{dx}=2x+3
\]

when \(x=2\), the derivative is \(7\).

In [ ]:
x = tf.Variable(2.0)

with tf.GradientTape() as tape:
    y = x**2 + 3*x + 2

gradient = tape.gradient(y, x)

print("Output:", y.numpy())
print("Gradient:", gradient.numpy())


### Simple gradient descent example

In [ ]:
w = tf.Variable(0.0)
learning_rate = 0.1

for epoch in range(50):
    with tf.GradientTape() as tape:
        loss = (w - 5.0) ** 2

    gradient = tape.gradient(loss, w)
    w.assign_sub(learning_rate * gradient)

print("Learned parameter:", w.numpy())
print("Final loss:", loss.numpy())


The learned parameter should approach `5`. `GradientTape` records operations involving watched variables; `tf.Variable` is useful for values that need to be updated during training.

## 4. Build your first neural network

We will build a small regression network:

- **Input:** two numerical features
- **Hidden layer:** eight neurons with ReLU activation
- **Output:** one numerical prediction

Keras layers track their trainable parameters automatically.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

simple_model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])

simple_model.summary()


### Make an initial prediction

In [ ]:
sample = tf.constant([[0.3, 0.8]], dtype=tf.float32)
prediction = simple_model(sample)

print("Prediction:", prediction.numpy().item())


The initial prediction is not meaningful yet because the model has not been trained.

## 5. Train a binary classifier on synthetic data

### Problem definition

Create two numerical features, `x1` and `x2`. The label is `1` if their sum is at least `1`; otherwise it is `0`.

This synthetic dataset lets us practice the full training workflow without downloading data.

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

X = tf.random.uniform((1000, 2), minval=0.0, maxval=1.0)
y = tf.cast(tf.reduce_sum(X, axis=1, keepdims=True) >= 1.0, tf.float32)

print("Input shape:", X.shape)
print("Target shape:", y.shape)
print("First five inputs:\n", X[:5].numpy())
print("First five labels:\n", y[:5].numpy())


### Split into training and testing sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X.numpy(),
    y.numpy(),
    test_size=0.2,
    random_state=42,
    stratify=y.numpy().ravel()
)

X_train = tf.convert_to_tensor(X_train_np, dtype=tf.float32)
X_test = tf.convert_to_tensor(X_test_np, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train_np, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test_np, dtype=tf.float32)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


### Configure the model

For binary classification, use one output logit and `BinaryCrossentropy(from_logits=True)`. This loss applies the sigmoid transformation internally in a numerically stable way, so the model should output raw logits.

In [ ]:
binary_model = keras.Sequential([
    keras.Input(shape=(2,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(1)
])

binary_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

binary_model.summary()


### Train the model

In [ ]:
history = binary_model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("Final training loss:", round(history.history["loss"][-1], 4))
print("Final training accuracy:", round(history.history["accuracy"][-1], 4))
print("Final validation accuracy:", round(history.history["val_accuracy"][-1], 4))


### What happens during training?

1. The model produces predictions (forward pass).
2. The loss function measures prediction error.
3. TensorFlow/Keras calculates gradients.
4. The optimizer updates trainable parameters.
5. The process repeats across batches and epochs.

An **epoch** is one pass through the training dataset. Validation data is used to monitor performance during training, not to update model weights.

### Plot training loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy loss")
plt.legend()
plt.grid(True)
plt.show()


## 6. Evaluate the model

A model should be tested on examples that were not used to fit its parameters. We will calculate accuracy, precision, recall, and a confusion matrix.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

test_logits = binary_model.predict(X_test, verbose=0)
test_probabilities = tf.math.sigmoid(test_logits).numpy().ravel()
test_predictions = (test_probabilities >= 0.5).astype(int)
y_true = y_test.numpy().ravel().astype(int)

print("Test accuracy:", round(accuracy_score(y_true, test_predictions), 4))
print("\nClassification report:")
print(classification_report(y_true, test_predictions, zero_division=0))
print("\nConfusion matrix:")
print(confusion_matrix(y_true, test_predictions))


- **Accuracy:** fraction of predictions that are correct.
- **Precision:** among predicted positives, the fraction that are truly positive.
- **Recall:** among actual positives, the fraction correctly found.
- **Confusion matrix:** counts correct and incorrect predictions by class.

Accuracy alone may be misleading when classes are imbalanced.

### Predict a new example

In [ ]:
sample = tf.constant([[0.3, 0.8]], dtype=tf.float32)
logit = binary_model(sample, training=False)
probability = tf.math.sigmoid(logit).numpy().item()
predicted_class = int(probability >= 0.5)

print("Probability of class 1:", round(probability, 4))
print("Predicted class:", predicted_class)


Since `0.3 + 0.8 >= 1`, the expected class under our synthetic rule is `1`.

## 7. Save and load a model

Keras supports saving the complete model, including architecture, weights, and compile configuration, using the `.keras` format.

In [ ]:
binary_model.save("tensorflow_binary_classifier.keras")
print("Model saved.")


### Load the saved model and compare predictions

In [ ]:
loaded_model = keras.models.load_model("tensorflow_binary_classifier.keras")

sample = tf.constant([[0.3, 0.8]], dtype=tf.float32)
original_prediction = binary_model(sample, training=False).numpy()
loaded_prediction = loaded_model(sample, training=False).numpy()

print("Original logit:", original_prediction.item())
print("Loaded logit:", loaded_prediction.item())
print("Predictions match:", np.allclose(original_prediction, loaded_prediction))


Keep the model's input feature order and any preprocessing steps consistent when using it in another application.

## 8. GPU detection and device placement

TensorFlow can use supported GPUs for many operations. The code below checks whether TensorFlow detects a GPU. If no supported GPU is available, TensorFlow normally runs on the CPU.

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus)

device = "/GPU:0" if gpus else "/CPU:0"
print("Selected device:", device)

with tf.device(device):
    a = tf.random.uniform((100, 100))
    b = tf.random.uniform((100, 100))
    c = tf.matmul(a, b)

print("Result shape:", c.shape)
print("Result device:", c.device)


A GPU does not guarantee faster execution for tiny workloads; setup and data-transfer overhead can outweigh the benefits. GPU availability depends on your operating system, hardware, and TensorFlow installation.

## 9. Mini-project: handwritten digit classification with MNIST

### Objective

Train a neural network to recognize handwritten digits from `0` to `9`. Each MNIST image is a 28 × 28 grayscale image.

`keras.datasets.mnist.load_data()` downloads the dataset if it is not already cached.

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)


### Normalize and display sample images

In [ ]:
# Pixel values are converted from [0, 255] to [0, 1].
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


### Build the digit classifier

The model flattens each image from 28 × 28 into 784 values, applies a hidden layer, and outputs ten class scores. The output layer has no softmax activation because `SparseCategoricalCrossentropy(from_logits=True)` expects logits and integer class labels.

In [ ]:
digit_model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10)
])

digit_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

digit_model.summary()


### Train the digit classifier

In [ ]:
digit_history = digit_model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)


### Evaluate on the test set

In [ ]:
test_loss, test_accuracy = digit_model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test accuracy percentage: {test_accuracy * 100:.2f}%")


### Display a prediction

In [ ]:
index = 0
image = x_test[index]
actual_label = int(y_test[index])

logits = digit_model.predict(image[np.newaxis, ...], verbose=0)
predicted_label = int(np.argmax(logits, axis=1)[0])

plt.imshow(image, cmap="gray")
plt.title(f"Actual: {actual_label} | Predicted: {predicted_label}")
plt.axis("off")
plt.show()


The actual accuracy depends on training, random initialization, software version, and hardware. This simple dense network should learn useful digit patterns, though convolutional neural networks are often better suited to image tasks.

## 10. Practice exercises

1. Create a tensor with shape `(5, 5)` and calculate its mean.
2. Change the hidden layer in the binary classifier from 8 to 32 neurons.
3. Compare ReLU and sigmoid in the hidden layer.
4. Try learning rates `0.1`, `0.01`, and `0.001`.
5. Compare training loss across 50 and 100 epochs.
6. Save and reload a model, then verify that predictions match.
7. Add a second hidden layer to the MNIST model.
8. Compare a dense MNIST model with a convolutional neural network.
9. Calculate a confusion matrix for MNIST predictions.
10. Check GPU availability and compare execution on supported hardware.

## 11. Interview questions

### Theoretical questions

1. What is TensorFlow?
2. What is the relationship between TensorFlow and Keras?
3. What is a tensor, and what does tensor rank mean?
4. What does `tf.GradientTape` do?
5. What is the difference between a tensor and a `tf.Variable`?
6. What is an epoch?
7. What is the purpose of a loss function?
8. What does an optimizer do?
9. Why use `BinaryCrossentropy(from_logits=True)`?
10. Why does `SparseCategoricalCrossentropy(from_logits=True)` work with integer labels and raw class scores?
11. What is the difference between training and validation data?
12. Why normalize image pixel values?

### Practical questions

1. Create and reshape tensors.
2. Use `tf.GradientTape` to calculate a derivative.
3. Build a neural network with `keras.Sequential`.
4. Train a binary classifier and plot its loss.
5. Evaluate a classifier with a confusion matrix.
6. Save and reload a `.keras` model.
7. Detect available GPUs.
8. Train an MNIST classifier.
9. Explain batch size and learning rate.
10. Describe methods to reduce overfitting, such as dropout, regularization, and early stopping.

## 12. Final summary

After completing this tutorial, you should be able to:

- Create and manipulate TensorFlow tensors.
- Calculate gradients with `tf.GradientTape`.
- Define neural networks using Keras layers.
- Train models using a loss function and optimizer.
- Evaluate classification performance.
- Save and reload trained models.
- Check GPU availability.
- Build an end-to-end MNIST digit classifier.

**Recommended next tutorial:** Convolutional Neural Networks (CNNs) in TensorFlow, followed by transfer learning and a practical computer-vision project.